# Week 2, Day 4: A WrestleMania calendar desk with two tools

**Lesson:** [week2/day4.ipynb](../../../week2/day4.ipynb), *Airline AI assistant: tools*. **About 40 minutes.**

**What you practice here**

- Describing a Python function to the model as a tool: its name, a description and JSON-schema parameters, in the `tools` list.
- Passing tools to a Chat Completions call, spotting a reply that asks for them, and reading the calls out of the SDK's message: each call's name, its JSON arguments and its id.
- Answering every call with a tool message and asking again, in a loop, until the model answers in text.
- The lesson's exercise: a second tool that changes data, as a tool to set a ticket's price would.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week2/day4", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson gives the airline assistant a tool, `get_ticket_price`, described to the model in the `tools` list. When
the model asks for the tool, `handle_tool_calls` runs it and `chat` sends the results back, in a loop, until the model
answers in text. The lesson's exercise is to add a tool that sets a price. Build the same pair for a WrestleMania
calendar desk: one tool looks up when a WrestleMania was held, and the other records a new date.

`chat` takes the client as its first parameter and makes the Chat Completions calls itself, and `handle_tool_calls`
reads the model's reply as the SDK gives it. The checks hand them a stand-in shaped like the real client. Its replies
ask for your tools the way the real API does, with the arguments as JSON text, and nothing is spent. The checks put
`DATES` back as it was after recording their test dates.

## What each call should give

```python
get_event_date("  wrestlemania iii ")    # '29 March 1987'
get_event_date("WrestleMania 99")        # 'unknown'
set_event_date("WrestleMania 50", "2 April 2034")
# 'Saved: WrestleMania 50 is on 2 April 2034'  (your own wording, holding the name and the date)

# a reply that asks for a tool has the finish reason 'tool_calls', and its message holds the calls:
handle_tool_calls(message)
# [{'role': 'tool', 'content': '29 March 1987', 'tool_call_id': 'call_1'}]

chat(client, "When was WrestleMania III?", [])
# asks with the tools, runs get_event_date for the model, asks again with the result,
# and returns the answer, such as 'WrestleMania III was held on 29 March 1987.'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import json

### 1 · `get_event_date(event)`

Looks up when a WrestleMania was held, in a dict named `DATES` that you keep in the same cell. `DATES` holds at least
these four, with the dates written exactly so: WrestleMania I on `"31 March 1985"`, WrestleMania III on
`"29 March 1987"`, WrestleMania X on `"20 March 1994"` and WrestleMania 32 on `"3 April 2016"`. The whole name must
match, ignoring upper and lower case and the whitespace around it: `" wrestlemania iii "` finds WrestleMania III, but
`"Mania III"` finds nothing. Return the date, or `"unknown"` for any other name.

<details><summary>Hint</summary>

Keep the names in `DATES` in lower case, and look up `event.strip().lower()`. `DATES.get(key, "unknown")` gives the
default.

</details>

In [ ]:
def get_event_date(event):
    # Write get_event_date here, as described above.
    raise NotImplementedError("get_event_date is not written yet")

In [ ]:
check("get_event_date")

### 2 · `set_event_date(event, date)`

Records the date of a WrestleMania in `DATES`: the lesson's exercise, a tool that changes data. Match the name as
`get_event_date` does, so that `get_event_date` finds the new date at once, and replace any date the event already
had. Return a short confirmation for the model that holds the event's name and the date, such as
`Saved: WrestleMania 50 is on 2 April 2034`.

<details><summary>Hint</summary>

`DATES[event.strip().lower()] = date`, then return an f-string.

</details>

In [ ]:
def set_event_date(event, date):
    # Write set_event_date here, as described above.
    raise NotImplementedError("set_event_date is not written yet")

In [ ]:
check("set_event_date")

### 3 · `TOOLS`

Describes both tools to the model: the list you pass as the tools of a Chat Completions call. It holds two entries in
the lesson's shape, a `"function"` entry around a description like its `price_function`: one for `get_event_date`,
with one parameter, `event`, and one for `set_event_date`, with the parameters `event` and `date`. Give each function
a description that tells the model when to use it. Give its parameters the lesson's JSON schema: an object whose
properties are strings, each with a description of its own, all of them required, and no others allowed.

<details><summary>Hint</summary>

One entry, in the lesson's shape:

```python
{"type": "function", "function": {
    "name": "get_event_date",
    "description": "...",
    "parameters": {"type": "object",
                   "properties": {"event": {"type": "string", "description": "..."}},
                   "required": ["event"],
                   "additionalProperties": False}}}
```

</details>

In [ ]:
TOOLS = []

In [ ]:
check("TOOLS")

### 4 · `handle_tool_calls(message)`

Runs the tools the model asked for, and returns one answer per call. `message` is the model's reply as the SDK gives
it: the first choice's message. For each of its tool calls, in order, read the call's function name and its arguments
(JSON text, so parse them), run `get_event_date` or `set_event_date`, and add a tool message that holds the result and
the call's id. A call to any other tool gets the content `"unknown tool"`, because every call needs an answer or the
API refuses the next request; the lesson's version skips such a call. Return the list.

<details><summary>Hint</summary>

```python
for call in message.tool_calls:
    arguments = json.loads(call.function.arguments)
    ...  # pick the tool by call.function.name
    answers.append({"role": "tool", "content": result, "tool_call_id": call.id})
```

</details>

In [ ]:
def handle_tool_calls(message):
    # Write handle_tool_calls here, as described above.
    raise NotImplementedError("handle_tool_calls is not written yet")

In [ ]:
check("handle_tool_calls")

### 5 · `chat(client, message, history, model="gpt-4.1-mini")`

Answers one message, running tools for as long as the model asks for them. Write `SYSTEM` in this cell too: tell the
assistant to use the tools for dates, and to say so when it does not know. Build the messages: a system message
holding `SYSTEM`, the history with only `role` and `content`, then a user message holding `message`. Make a Chat
Completions call with the model `model`, those messages and `TOOLS` as the tools. While the reply asks for tools, its
finish reason being `"tool_calls"`, add the reply's message to the messages as it is, then the answers from
`handle_tool_calls`, and call again with the same tools, as the lesson's last `chat` does. Return the last reply's
text, and leave the history unchanged.

<details><summary>Hint</summary>

```python
response = client.chat.completions.create(model=model, messages=messages, tools=TOOLS)
while response.choices[0].finish_reason == "tool_calls":
    reply = response.choices[0].message
    messages.append(reply)
    messages.extend(handle_tool_calls(reply))
    response = client.chat.completions.create(model=model, messages=messages, tools=TOOLS)
```

</details>

In [ ]:
def chat(client, message, history, model="gpt-4.1-mini"):
    # Write chat here, as described above.
    raise NotImplementedError("chat is not written yet")

In [ ]:
check("chat")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 2, Day 4 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

A question to `gpt-4.1-mini` costs a fraction of a cent, a little more when it uses a tool, as the model then answers
twice.

1. Load your key with `load_dotenv(override=True)`, make `openai = OpenAI()`, and print
   `chat(openai, "When was WrestleMania III?", [])`.
2. Ask about two events at once, such as WrestleMania X and WrestleMania 32: the model may ask for both in one reply.
3. Announce a date ("WrestleMania 50 will be on 2 April 2034, please note it"), then print
   `get_event_date("WrestleMania 50")` to see that the tool changed your data.
4. Chat in a page: `gr.ChatInterface(fn=respond, type="messages").launch()`, after `import gradio as gr`, where
   `respond(message, history)` returns `chat(openai, message, history)`.
5. Optional: move the dates into SQLite, as the lesson does with `prices.db`: a table keyed by the event's name, with
   `get_event_date` and `set_event_date` reading and writing it; or run the chat on `llama3.2` through Ollama.

<details><summary>Stuck? One way to write it</summary>

```python
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai = OpenAI()
print(chat(openai, "When was WrestleMania III?", []))
print(chat(openai, "When were WrestleMania X and WrestleMania 32?", []))
print(chat(openai, "WrestleMania 50 will be on 2 April 2034. Please note it.", []))
print(get_event_date("WrestleMania 50"))


def respond(message, history):
    return chat(openai, message, history)


gr.ChatInterface(fn=respond, type="messages").launch()
```

```python
# Optional: the dates in SQLite, as the lesson keeps its prices in prices.db
import sqlite3
import tempfile
from pathlib import Path

DB = str(Path(tempfile.mkdtemp()) / "dates.db")
UPSERT = "INSERT INTO dates (event, date) VALUES (?, ?) ON CONFLICT(event) DO UPDATE SET date = ?"
with sqlite3.connect(DB) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS dates (event TEXT PRIMARY KEY, date TEXT)")
    for name, date in DATES.items():
        conn.execute(UPSERT, (name, date, date))


def get_event_date(event):
    with sqlite3.connect(DB) as conn:
        row = conn.execute("SELECT date FROM dates WHERE event = ?", (event.strip().lower(),)).fetchone()
    return row[0] if row else "unknown"


def set_event_date(event, date):
    with sqlite3.connect(DB) as conn:
        conn.execute(UPSERT, (event.strip().lower(), date, date))
    return f"Saved: {event.strip()} is on {date}"


print(chat(openai, "When was WrestleMania 32?", []))

# Optional: the same desk on a local model, with Ollama running
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
print(chat(ollama, "When was WrestleMania X?", [], model="llama3.2"))
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.